[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/neoplasia-detection/blob/develop/RARE2026_V2.ipynb)

# RARE2026 V2 — End-to-End Training Pipeline

**Task:** Binary classification (neoplasia vs. NDBE) on Barrett's esophagus endoscopy images.  
**Metric:** PPV @ 90% Recall — bootstrap-simulated at 1% clinical prevalence.  
**Branch:** `develop`

| Section | Content |
|---|---|
| 1 | Environment setup |
| 2 | BONSAI dataset — download, EDA, patient-aware splits |
| 3 | EVC Barretts FullSet — enrichment of val_calibration |
| 4 | Dataset integration |
| 5 | Repository + GastroNet weights |
| 6 | Training — ViT-B DINOv2 ensemble (5 seeds) |
| 7 | Training — SSL diversity ensemble (ResNet50 × 4 variants × 5 seeds) |
| 8 | Calibration affine + bootstrap evaluation |
| 9 | PatchCore anomaly detector fitting |
| 10 | manifest.json generation |
| 11 | Inference demo (EnsemblePredictor + PatchCore) |
| 12 | Download outputs |

---
## Section 1 — Environment Setup

In [ ]:
%pip install -q peft>=0.11.1 timm gdown
%pip install -q psrcal>=1.0.0 opencv-python-headless
%pip install -q SimpleITK hydra-core omegaconf scikit-learn tqdm

In [ ]:
import os, torch

REPO_DIR    = '/root/rare26'
DATA_DIR    = '/root/data'
EVC_DIR     = '/root/EVC_Barretts_FullSet'
WEIGHTS_DIR = '/root/rare26/weights'
OUTPUT_DIR  = '/root/outputs'

for d in [DATA_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')
else:
    print('No GPU — check Runtime > Change runtime type')

---
## Section 2 — BONSAI Dataset

3 095 images · 158 neoplasia · 2 937 NDBE · 2 centres (Olympus endoscopes).

In [ ]:
import os, zipfile, gdown
from pathlib import Path

BONSAI_ZIP     = '/root/bonsai_dataset.zip'
BONSAI_EXTRACT = '/root/bonsai_raw'
BONSAI_FILE_ID = '1Hs9O6Gckq3CUuPMN5Symq5roQahreYnx'

if not Path(BONSAI_EXTRACT).exists():
    gdown.download(f'https://drive.google.com/uc?id={BONSAI_FILE_ID}', BONSAI_ZIP, quiet=False)
    os.makedirs(BONSAI_EXTRACT, exist_ok=True)
    with zipfile.ZipFile(BONSAI_ZIP) as z:
        z.extractall(BONSAI_EXTRACT)
    print('Extracted.')
else:
    print('Already extracted.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

EXTS = {'.jpg', '.jpeg', '.png'}
rows = []
for root, _, files in os.walk(BONSAI_EXTRACT):
    imgs = [f for f in files if Path(f).suffix.lower() in EXTS]
    if not imgs: continue
    parts = Path(root).relative_to(BONSAI_EXTRACT).parts
    center  = parts[0] if len(parts) > 0 else 'unknown'
    cls_raw = parts[1] if len(parts) > 1 else 'unknown'
    for f in imgs:
        rows.append({'image_path': os.path.join(root, f), 'center': center, 'class_raw': cls_raw})

bonsai_df = pd.DataFrame(rows)
bonsai_df['label'] = (bonsai_df['class_raw'] != 'ndbe').astype(int)

print(f'Total: {len(bonsai_df)} images')
print(bonsai_df.groupby(['center', 'class_raw'])['label'].count().to_string())

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=bonsai_df, x='center', hue='class_raw', ax=ax)
ax.set_title('Images per centre')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

os.makedirs(DATA_DIR, exist_ok=True)
bonsai_df[['image_path', 'label']].to_csv(f'{DATA_DIR}/bonsai_all.csv', index=False)

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(sss1.split(np.zeros(len(bonsai_df)), bonsai_df['label'].values))

train_df = bonsai_df.iloc[train_idx][['image_path', 'label']].reset_index(drop=True)
val_df   = bonsai_df.iloc[val_idx][['image_path', 'label']].reset_index(drop=True)

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
sel_idx, cal_idx = next(sss2.split(np.zeros(len(val_df)), val_df['label'].values))

val_sel_df = val_df.iloc[sel_idx].reset_index(drop=True)
val_cal_df = val_df.iloc[cal_idx].reset_index(drop=True)

train_df.to_csv(f'{DATA_DIR}/train.csv',             index=False)
val_sel_df.to_csv(f'{DATA_DIR}/val_selection.csv',   index=False)
val_cal_df.to_csv(f'{DATA_DIR}/val_calibration.csv', index=False)

for name, df in [('train', train_df), ('val_selection', val_sel_df), ('val_calibration', val_cal_df)]:
    print(f'{name:<22}: {len(df):5d} images  ({df["label"].sum()} pos)')

---
## Section 3 — EVC Barretts FullSet

External dataset with `patXX_imY_DIAGNOSIS` filename convention — patient ID is explicit.  
Used to enrich `val_calibration` to ≥ 50 positives for robust bootstrap evaluation.

In [ ]:
import os, zipfile, gdown
from pathlib import Path

EVC_ZIP     = '/root/EVC_Barretts_FullSet.zip'
EVC_FILE_ID = '1KTeQL8vcLXDxCnCM09VGxN5Sm_wv1d2h'

if not Path(EVC_DIR).exists():
    gdown.download(f'https://drive.google.com/uc?id={EVC_FILE_ID}', EVC_ZIP, quiet=False)
    os.makedirs(EVC_DIR, exist_ok=True)
    with zipfile.ZipFile(EVC_ZIP) as z:
        z.extractall(EVC_DIR)
    print('Extracted.')
else:
    print('Already extracted.')

EVC_AVAILABLE = Path(EVC_DIR).exists()
print(f'EVC available: {EVC_AVAILABLE}')

In [ ]:
if EVC_AVAILABLE:
    import pandas as pd
    from pathlib import Path

    rows = []
    for root, _, files in os.walk(EVC_DIR):
        for f in files:
            stem  = Path(f).stem
            parts = stem.split('_')
            ext   = Path(f).suffix.lower()
            if len(parts) >= 3 and ext not in ('.bmp', '.mat'):
                rows.append({
                    'image_path': os.path.join(root, f),
                    'patient_id': parts[0],
                    'diagnosis':  parts[2],
                })

    evc_df = pd.DataFrame(rows)
    evc_df['label'] = (evc_df['diagnosis'] == 'ACHD').astype(int)
    print(f'EVC images: {len(evc_df)}')
    print(evc_df.groupby('diagnosis')['label'].count().to_string())

---
## Section 4 — Dataset Integration

Merge BONSAI train with EVC train subset. Enrich val_calibration with EVC positives.

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

train_df   = pd.read_csv(f'{DATA_DIR}/train.csv')
val_cal_df = pd.read_csv(f'{DATA_DIR}/val_calibration.csv')

if EVC_AVAILABLE:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
    evc_clean = evc_df[['image_path', 'label', 'patient_id']]
    train_idx, cal_idx = next(gss.split(evc_clean, groups=evc_clean['patient_id']))

    evc_train = evc_clean.iloc[train_idx][['image_path', 'label']]
    evc_cal   = evc_clean.iloc[cal_idx][['image_path', 'label']]

    train_merged     = pd.concat([train_df, evc_train], ignore_index=True)
    val_cal_enriched = pd.concat([val_cal_df, evc_cal],  ignore_index=True)

    TRAIN_CSV   = f'{DATA_DIR}/train_merged.csv'
    VAL_CAL_CSV = f'{DATA_DIR}/val_calibration_enriched.csv'
    train_merged.to_csv(TRAIN_CSV,   index=False)
    val_cal_enriched.to_csv(VAL_CAL_CSV, index=False)
else:
    TRAIN_CSV   = f'{DATA_DIR}/train.csv'
    VAL_CAL_CSV = f'{DATA_DIR}/val_calibration.csv'
    train_merged     = train_df
    val_cal_enriched = val_cal_df

VAL_SEL_CSV = f'{DATA_DIR}/val_selection.csv'
val_sel_df  = pd.read_csv(VAL_SEL_CSV)

print('Final dataset:')
for name, df in [('train', train_merged), ('val_selection', val_sel_df), ('val_calibration', val_cal_enriched)]:
    print(f'  {name:<22}: {len(df):5d} images  ({df["label"].sum()} pos)')

---
## Section 5 — Repository + GastroNet Weights

In [ ]:
import os, sys
from pathlib import Path

if not Path(f'{REPO_DIR}/scripts/train.py').exists():
    !git clone -b develop https://github.com/hashirama21/neoplasia-detection.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin develop

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
!pip install -q -r requirements.txt
print(f'Working directory: {os.getcwd()}')

In [ ]:
import torch, gdown
from pathlib import Path

Path(WEIGHTS_DIR).mkdir(parents=True, exist_ok=True)

GASTRONET_ID   = '1evpQ2myroMVVhmZV9M_tW-v4G0aXbmVW'
GASTRONET_PATH = Path(WEIGHTS_DIR) / 'dinov2_gastronet5m.pth'

if not GASTRONET_PATH.exists():
    gdown.download(f'https://drive.google.com/uc?id={GASTRONET_ID}', str(GASTRONET_PATH), quiet=False)

ckpt = torch.load(GASTRONET_PATH, map_location='cpu', weights_only=True)
state = ckpt.get('teacher', ckpt)
n_params = sum(v.numel() for v in state.values() if isinstance(v, torch.Tensor))
print(f'GastroNet-5M: {GASTRONET_PATH.stat().st_size/1e6:.0f} MB  |  {n_params/1e6:.1f}M params')

In [ ]:
import gdown
from pathlib import Path

RN50_WEIGHTS = {
    'RN50_GastroNet-5M_DINOv1.pth':                          '1dL7d1O8xJ8riyQ7xmLoQC6oRNYl_e_py',
    'RN50_GastroNet-5M_MOCOv2.pth':                          '1HDdB3EVmV4BJj8ovjgVuJh-1rgVUYhCa',
    'RN50_GastroNet-5M_SIMCLRv2.pth':                        '1aaZNUL2PWFgaDZV4BtfHtZh5TzrC2Pr0',
    'RN50_Billion-Scale-SWSL%2BGastroNet-5M_DINOv1.pth':     '13u2oUmHW78j4TOUUpl4hseel4i5AhhVe',
}

for fname, file_id in RN50_WEIGHTS.items():
    dest = Path(WEIGHTS_DIR) / fname
    if not dest.exists():
        print(f'Downloading {fname}...')
        gdown.download(f'https://drive.google.com/uc?id={file_id}', str(dest), quiet=True)
    else:
        print(f'Already present: {fname}')

---
## Section 6 — Training: ViT-B DINOv2 Ensemble (5 seeds)

| Backbone | LoRA | LR backbone | LR head | Loss | Seeds |
|---|---|---|---|---|---|
| DINOv2 ViT-B/14 + GastroNet-5M | rank=8 α=16 | 1e-5 | 1e-3 | ASL γ⁻=4 | 42, 123, 456, 789, 1337 |

~7 min/seed on L4 GPU.

In [ ]:
import sys, site
os.environ['PYTHONPATH'] = f"{REPO_DIR}:{':'.join(site.getsitepackages())}"

SEEDS = [42, 123, 456, 789, 1337]

for seed in SEEDS:
    seed_out = f'{OUTPUT_DIR}/dinov2_seed_{seed}'
    print(f'\n[Seed {seed}]')
    !python {REPO_DIR}/scripts/train.py \
        project.output_dir={seed_out} \
        project.seed={seed} \
        model=dinov2_gastronet \
        calibration=affine \
        paths.data_dir={DATA_DIR} \
        paths.weights_dir={WEIGHTS_DIR} \
        device={device} \
        num_workers=2 \
        training.cross_validation.enabled=false \
        data.train_csv={TRAIN_CSV} \
        data.val_calibration_csv={VAL_CAL_CSV} \
        data.val_selection_csv={VAL_SEL_CSV}

print('\nViT-B training complete.')

In [ ]:
import json, pandas as pd
from pathlib import Path

rows = []
for seed in SEEDS:
    cal_file = Path(OUTPUT_DIR) / f'dinov2_seed_{seed}' / 'results' / 'calibration_results.json'
    if cal_file.exists():
        r = json.loads(cal_file.read_text())
        rows.append({'seed': seed, 'model': 'dinov2', **{k: round(v, 4) for k, v in r.items() if isinstance(v, float)}})
    else:
        rows.append({'seed': seed, 'model': 'dinov2', 'status': 'missing'})

from IPython.display import display
display(pd.DataFrame(rows))

---
## Section 7 — Training: SSL Diversity Ensemble (ResNet50 × 4 variants × 5 seeds)

**Innovation:** Three SSL pretraining methods (DINOv1, MOCOv2, SIMCLRv2) on the same backbone
produce decorrelated representations — errors are structurally different, improving Noisy-OR fusion.

20 models total. ~4 min/model on L4. Estimated total: ~80 min.

> Set `TRAIN_SSL_DIVERSITY = False` to skip this section.

In [ ]:
TRAIN_SSL_DIVERSITY = True

SSL_CHECKPOINTS = {
    'dino':   'RN50_GastroNet-5M_DINOv1.pth',
    'moco':   'RN50_GastroNet-5M_MOCOv2.pth',
    'simclr': 'RN50_GastroNet-5M_SIMCLRv2.pth',
    'swsl':   'RN50_Billion-Scale-SWSL%2BGastroNet-5M_DINOv1.pth',
}

if TRAIN_SSL_DIVERSITY:
    for ssl_name, ckpt_fname in SSL_CHECKPOINTS.items():
        ckpt_path = f'{WEIGHTS_DIR}/{ckpt_fname}'
        if not os.path.exists(ckpt_path):
            print(f'Skipping {ssl_name} — checkpoint not found: {ckpt_path}')
            continue
        for seed in SEEDS:
            seed_out = f'{OUTPUT_DIR}/rn50_{ssl_name}_seed_{seed}'
            print(f'[rn50_{ssl_name} seed={seed}]')
            !python {REPO_DIR}/scripts/train.py \
                project.output_dir={seed_out} \
                project.seed={seed} \
                model=rn50_gastronet \
                training=rn50_base \
                calibration=affine \
                model.checkpoint_path={ckpt_path} \
                paths.data_dir={DATA_DIR} \
                paths.weights_dir={WEIGHTS_DIR} \
                device={device} \
                num_workers=2 \
                data.train_csv={TRAIN_CSV} \
                data.val_calibration_csv={VAL_CAL_CSV} \
                data.val_selection_csv={VAL_SEL_CSV}
    print('SSL diversity training complete.')
else:
    print('SSL diversity skipped (TRAIN_SSL_DIVERSITY = False).')

---
## Section 8 — Calibration Affine + Bootstrap Evaluation

Evaluates the **ensemble** (all checkpoints together) on `val_selection`.  
Affine calibration encodes the 1% test prevalence directly as `priors=[100/101, 1/101]`.

In [ ]:
!python {REPO_DIR}/scripts/evaluate.py \
    project.output_dir={OUTPUT_DIR} \
    paths.data_dir={DATA_DIR} \
    paths.weights_dir={WEIGHTS_DIR} \
    device={device} \
    num_workers=2 \
    calibration=affine

In [ ]:
import json, pandas as pd
from pathlib import Path
from IPython.display import display

eval_path = Path(OUTPUT_DIR) / 'ensemble' / 'results' / 'evaluation_results.json'
if eval_path.exists():
    ev = json.loads(eval_path.read_text())
    rows = [
        ('Median PPV@90Recall', ev.get('median_ppv', 'N/A')),
        ('Std PPV',             ev.get('std_ppv', 'N/A')),
        ('P10 PPV',             ev.get('p10_ppv', 'N/A')),
        ('Median Recall',       ev.get('median_recall', 'N/A')),
        ('Threshold',           ev.get('optimal_threshold', 'N/A')),
    ]
    df = pd.DataFrame(rows, columns=['Metric', 'Value'])
    df['Value'] = df['Value'].apply(lambda x: f'{x:.4f}' if isinstance(x, float) else str(x))
    display(df.set_index('Metric'))

    std = ev.get('std_ppv', 999)
    rec = ev.get('median_recall', 0)
    p10 = ev.get('p10_ppv', 0)
    for ok, label in [
        (std < 0.15,  f'PPV std < 0.15       : {std:.4f}'),
        (rec >= 0.88, f'Recall >= 0.88       : {rec:.4f}'),
        (p10 > 0.30,  f'P10 PPV > 0.30       : {p10:.4f}'),
    ]:
        print(f'  {"✓" if ok else "✗"} {label}')
else:
    print(f'No results at {eval_path} — run evaluate cell first.')

---
## Section 9 — PatchCore Anomaly Detector

Builds a memory bank of patch features from **normal (NDBE)** training images.  
At inference, frames whose patches deviate from this bank get an anomaly score  
fused with the ensemble score via Noisy-OR.  

> Set `FIT_PATCHCORE = False` to skip.

In [ ]:
import glob
from pathlib import Path

FIT_PATCHCORE = True

if FIT_PATCHCORE:
    all_ckpts = sorted(
        glob.glob(f'{OUTPUT_DIR}/dinov2_seed_42/checkpoints/*.pt'),
        key=lambda p: float(Path(p).stem.rsplit('_', 1)[-1]),
        reverse=True,
    )
    if not all_ckpts:
        print('No ViT-B checkpoint found — run Section 6 first.')
        FIT_PATCHCORE = False
    else:
        best_ckpt = all_ckpts[0]
        print(f'Using checkpoint: {Path(best_ckpt).name}')
        !python {REPO_DIR}/scripts/fit_patchcore.py \
            --checkpoint {best_ckpt} \
            --model-config {REPO_DIR}/configs/model/dinov2_gastronet.yaml \
            --train-csv {TRAIN_CSV} \
            --val-csv {VAL_CAL_CSV} \
            --out-dir {WEIGHTS_DIR} \
            --device {device}
        print('PatchCore memory bank saved to weights/patchcore.pkl')
else:
    print('PatchCore skipped.')

---
## Section 10 — Manifest Generation

`manifest.json` tells `inference.py` which checkpoint uses which model config.  
This cell selects the best checkpoint per seed and generates the manifest.

In [ ]:
import glob, shutil, json
from pathlib import Path

def best_checkpoint(run_dir: str) -> str | None:
    ckpts = sorted(
        glob.glob(f'{run_dir}/checkpoints/*.pt'),
        key=lambda p: float(Path(p).stem.rsplit('_', 1)[-1]),
        reverse=True,
    )
    return ckpts[0] if ckpts else None

manifest = []

for seed in SEEDS:
    run_dir = f'{OUTPUT_DIR}/dinov2_seed_{seed}'
    ckpt = best_checkpoint(run_dir)
    if ckpt:
        dest = f'{WEIGHTS_DIR}/dinov2_seed{seed}_{Path(ckpt).name}'
        shutil.copy(ckpt, dest)
        manifest.append({'checkpoint': Path(dest).name, 'model_config': 'dinov2_gastronet'})
        print(f'✓ dinov2 seed={seed}  →  {Path(dest).name}')

if TRAIN_SSL_DIVERSITY:
    for ssl_name in SSL_CHECKPOINTS:
        for seed in SEEDS:
            run_dir = f'{OUTPUT_DIR}/rn50_{ssl_name}_seed_{seed}'
            ckpt = best_checkpoint(run_dir)
            if ckpt:
                dest = f'{WEIGHTS_DIR}/rn50_{ssl_name}_seed{seed}_{Path(ckpt).name}'
                shutil.copy(ckpt, dest)
                manifest.append({'checkpoint': Path(dest).name, 'model_config': 'rn50_gastronet'})
                print(f'✓ rn50_{ssl_name} seed={seed}  →  {Path(dest).name}')

manifest_path = f'{WEIGHTS_DIR}/manifest.json'
Path(manifest_path).write_text(json.dumps(manifest, indent=2))
print(f'\nmanifest.json: {len(manifest)} checkpoints → {manifest_path}')

# Also copy calibration artifacts
import os
cal_src = Path(OUTPUT_DIR) / 'ensemble' / 'results'
cal_dst = Path(WEIGHTS_DIR) / 'calibration'
cal_dst.mkdir(exist_ok=True)
for f in cal_src.glob('*'):
    shutil.copy(f, cal_dst / f.name)
    print(f'Copied calibration artifact: {f.name}')

---
## Section 11 — Inference Demo

Uses the full `EnsemblePredictor` (Noisy-OR + affine calibration + optional PatchCore).

In [ ]:
import sys, glob, random, json
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from PIL import Image
from omegaconf import OmegaConf

sys.path.insert(0, REPO_DIR)
from src.inference.predictor import EnsemblePredictor

device_t = torch.device(device)

predictor_cfg = OmegaConf.create({
    'tta': {'enabled': True},
    'ensemble': {'aggregation': 'noisy_or', 'uncertainty_penalty': 0.0},
    'threshold': 0.5,
})
predictor = EnsemblePredictor(predictor_cfg, device_t)

manifest = json.loads(Path(f'{WEIGHTS_DIR}/manifest.json').read_text())
configs_dir = Path(REPO_DIR) / 'configs' / 'model'
for entry in manifest:
    model_cfg = OmegaConf.load(configs_dir / f"{entry['model_config']}.yaml")
    model_cfg.checkpoint_path = ''
    predictor.load_model(f"{WEIGHTS_DIR}/{entry['checkpoint']}", model_cfg)

cal_path = Path(WEIGHTS_DIR) / 'calibration'
affine_path   = cal_path / 'affine_calibrator.pt'
isotonic_path = cal_path / 'isotonic_calibrator.pkl'
results_path  = cal_path / 'calibration_results.json'

if affine_path.exists():
    from src.calibration.calibrator import AffineCalibrator
    cal = AffineCalibrator(); cal.load(str(affine_path))
    predictor.calibrator = cal
elif isotonic_path.exists():
    from src.calibration.calibrator import IsotonicCalibrator
    cal = IsotonicCalibrator(); cal.load(str(isotonic_path))
    predictor.calibrator = cal

if results_path.exists():
    predictor.threshold = float(json.loads(results_path.read_text()).get('optimal_threshold', 0.5))

import torchvision.transforms as T
transform = T.Compose([
    T.Resize(448, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(392),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

all_imgs = (
    glob.glob(f'{BONSAI_EXTRACT}/**/*.jpg', recursive=True) +
    glob.glob(f'{BONSAI_EXTRACT}/**/*.png', recursive=True)
)
samples = random.sample(all_imgs, min(4, len(all_imgs)))

fig, axes = plt.subplots(1, len(samples), figsize=(16, 4))
if len(samples) == 1: axes = [axes]

for ax, img_path in zip(axes, samples):
    img = Image.open(img_path).convert('RGB')
    result = predictor.predict_single(img, transform)
    pred   = result['prediction']
    true_cls = 'neo' if Path(img_path).parent.name.lower() not in ('ndbe', 'ndbt') else 'ndbe'
    ax.imshow(img)
    ax.set_title(
        f'True: {true_cls}\nProb: {result["calibrated_prob"]:.3f}  unc: {result["uncertainty"]:.3f}\n→ {"NEOPLASIA" if pred else "NDBE"}',
        color='red' if pred else 'green', fontsize=9,
    )
    ax.axis('off')

plt.suptitle(f'Inference demo — {len(predictor.models)} models · Noisy-OR · thr={predictor.threshold:.3f}', fontsize=11)
plt.tight_layout(); plt.show()

---
## Section 12 — Download Outputs

In [ ]:
import os
os.environ['GDRIVE_ACCESS_TOKEN'] = ''  # paste output of: !gcloud auth print-access-token
!python {REPO_DIR}/scripts/download.py